#  Sentiment Analysis using NLP Pipeline & Machine Learning Models
**Internship Task:** Data Science Internship – February 2026  
**Objective:** Build an end-to-end Sentiment Analysis system using NLP preprocessing and multiple ML models.  
**Dataset:** IMDb Reviews Dataset (CSV)

##  Step 1 - Data Understanding (Load & Explore Dataset)


In [3]:
import pandas as pd

# Load dataset (replace with your downloaded dataset path)
df = pd.read_csv("IMDB Dataset.csv")

# Number of samples
print("Total Reviews:", df.shape[0])

# Class distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

# Sample reviews
df['review'].sample(5)

Total Reviews: 50000

Class Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


24155    Well, i must admit, when i saw the trailer for...
45646    On Sunday July 27, 1997, the first episode of ...
31851    Besides the comments on the technical merits o...
3286     I saw this movie when it aired on the WB and f...
31586    I tivo'd this on Turner Classic just because i...
Name: review, dtype: object

#  Step 2: NLP Preprocessing

We will:
- Lowercase all text
- Remove punctuation
- Remove stopwords
- Tokenize text
- Lemmatize words
- Handle URLs, emails, numbers, repeated characters
We will define a reusable function `preprocess_text`.

In [5]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+|\S+@\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    
    tokens = nltk.word_tokenize(text)
    tokens = [word for word in tokens if (word not in stop_words or word in ['no','not'])]
    tokens = [word for word in tokens if len(word) > 2 or word in ['no','not']]
    tokens = [re.sub(r'(.)\1{2,}', r'\1', word) for word in tokens]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    clean_sentence = ' '.join(tokens)
    return tokens, clean_sentence

# Test preprocessing
sample_text = df['review'][0]
tokens, clean_sentence = preprocess_text(sample_text)
print("Original Review:\n", sample_text)
print("\nTokens:\n", tokens)
print("\nCleaned Review:\n", clean_sentence)

[nltk_data] Downloading package punkt to C:\Users\karth/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\karth/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\karth/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Original Review:
 One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show

#  Step 3: Feature Engineering

 converting text to numeric features using:
1. Bag of Words (CountVectorizer)
2. TF-IDF Vectorizer

In [11]:
# Handle missing values first (VERY IMPORTANT)
df['review'] = df['review'].fillna("")

# Apply preprocessing safely
tokens_list = []
clean_list = []

for text in df['review']:
    tokens, clean = preprocess_text(text)
    tokens_list.append(tokens)
    clean_list.append(clean)

# Add to dataframe
df['tokens'] = tokens_list
df['clean_review'] = clean_list

# Check output
df[['review', 'clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching episode youll ...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


#  Step 4: Model Building

 training 3 models:
- Logistic Regression
- Naive Bayes
- Decision Tree

We will split data into training and testing sets.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklea rn.tree import DecisionTreeClassifier

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Initialize models
lr = LogisticRegression(max_iter=500)
nb = MultinomialNB()
dt = DecisionTreeClassifier()

# Train models
lr.fit(X_train, y_train)
nb.fit(X_train, y_train)
dt.fit(X_train, y_train)

DecisionTreeClassifier()

#  Step 5: Model Evaluation

We will evaluate models using:
- Accuracy
- Precision
- Recall
- F1 Score
- Confusion Matrix

In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    
print("Logistic Regression Performance:")
evaluate_model(lr, X_test, y_test)
print("\nNaive Bayes Performance:")
evaluate_model(nb, X_test, y_test)
print("\nDecision Tree Performance:")
evaluate_model(dt, X_test, y_test)

Logistic Regression Performance:
Accuracy: 0.893
Precision: 0.8833301139656171
Recall: 0.9075213335979361
F1 Score: 0.8952623335943618
Confusion Matrix:
 [[4357  604]
 [ 466 4573]]

Naive Bayes Performance:
Accuracy: 0.8679
Precision: 0.8842496899545267
Recall: 0.8489779718198055
F1 Score: 0.8662549357092234
Confusion Matrix:
 [[4401  560]
 [ 761 4278]]

Decision Tree Performance:
Accuracy: 0.72
Precision: 0.7233193696389387
Recall: 0.7195872196864457
F1 Score: 0.7214484679665738
Confusion Matrix:
 [[3574 1387]
 [1413 3626]]


#  step 6 : Comparison & Insights


### Model Performance Comparison

After training and evaluating three machine learning models, the following observations were made:

#### 1. Logistic Regression
- Achieved the highest accuracy of **89.3%**
- Balanced precision (**0.88**) and recall (**0.90**)
- Strong F1-score (**0.89**)
- Performed consistently well across both classes

#### 2. Naive Bayes
- Accuracy of **86.79%**
- Slightly lower recall compared to Logistic Regression
- Good performance but less balanced
- Works well for text data but may oversimplify relationships

#### 3. Decision Tree
- Lowest accuracy of **72%**
- Higher number of misclassifications (visible in confusion matrix)
- Prone to overfitting on training data
- Not ideal for high-dimensional text features

---

###  Best Model

**Logistic Regression performed the best overall**, providing the highest accuracy and balanced evaluation metrics. It handles high-dimensional sparse data (like TF-IDF features) effectively.

---

###  Observations

- TF-IDF vectorization significantly improved model performance compared to simple frequency-based approaches.
- Preprocessing steps like stopword removal, lemmatization, and noise cleaning helped in improving accuracy.
- Decision Tree struggled due to the complexity and sparsity of text features.

---

###  Trade-offs

- Logistic Regression: Best performance but slightly more computational cost
- Naive Bayes: Faster and simpler but slightly less accurate
- Decision Tree: Easy to interpret but poor performance on text data

---

###  Conclusion

For sentiment analysis tasks, **Logistic Regression with TF-IDF features is the most effective approach** among the tested models. Proper preprocessing plays a crucial role in achieving good performance.